## Train the LightGBM model

### 1. Hyperparameter tuning

Each Optuna trial picks a value for every hyperparameter, trains the model on the 3 validation folds,
and returns the mean Poisson deviance (lower is better). <br> Optuna uses the results to pick better values in the next trials.

The test set is not used here.

In [ ]:
import sys
from pathlib import Path

import optuna

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utilities.lightgbm_model import *
from utilities.training import *
from utilities.utilities import *
from utilities.values import *

data_dir = '../data/'
model_dir = '../models/'

# read the train-validation set (Jan 2020 - Feb 2021)
train_validation_df = read_parquet_file(data_dir + 'processed/train_validation.parquet')

In [ ]:
def objective(trial):
    # 1. let optuna pick the hyperparams for this trial
    hyperparams = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log = True),      # search on a log scale
        'n_estimators': trial.suggest_int('n_estimators', 100, 800, step = 50),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255, log = True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 300, log = True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log = True),
    }

    # 2. train a new model with these hyperparams on each validation fold, and score it
    scores_df = cross_validate(LightGBMModel, hyperparams, train_validation_df)
    mean_scores = scores_df[['MAE', 'RMSE', 'poisson_deviance']].mean()

    # Optuna tunes on the Poisson deviance only
    # MAE and RMSE are stored with the trial so they also appear in the results table
    trial.set_user_attr('MAE', mean_scores['MAE'])
    trial.set_user_attr('RMSE', mean_scores['RMSE'])

    # show the scores of this trial
    print('Poisson deviance per fold:', scores_df['poisson_deviance'].round(4).tolist())
    print(f'Trial {trial.number}: mean Poisson deviance = {mean_scores["poisson_deviance"]:.4f}, '
          f'mean MAE = {mean_scores["MAE"]:.4f}, mean RMSE = {mean_scores["RMSE"]:.4f}\n')
    return mean_scores['poisson_deviance']


# hide Optuna's own log line for every trial (the progress bar below shows the progress instead)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# the seed makes the search repeatable
study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler(seed = OPTUNA_SEED))

In [ ]:
n_trials = 30

# the progress bar shows how many trials are done, the time remaining and the best score so far
study.optimize(objective, n_trials = n_trials, show_progress_bar = True)

print('Best hyperparameters:', study.best_params)
print('Best mean Poisson deviance:', round(study.best_value, 4))
print('Mean MAE of the best trial:', round(study.best_trial.user_attrs['MAE'], 4))
print('Mean RMSE of the best trial:', round(study.best_trial.user_attrs['RMSE'], 4))

In [ ]:
# all trials, best first
trials_df = study.trials_dataframe()
trials_df.sort_values('value').head(10)

### 2. Validation predictions (for the ensemble)

- Get the best set of hyperparameters from optuna.
- For each of the cross-validation sets, train the model on the train set, and store the validation set predictions.
- These validation set predictions will be used to train the ensemble model later.

In [ ]:
# get the predictions of each validation fold
fold_results = get_out_of_fold_predictions(LightGBMModel, study.best_params, train_validation_df)

# save the predictions in each file separately
for fold_number, fold_df in enumerate(fold_results, start = 1):
    save_to_parquet_file(fold_df, data_dir + f'predictions/validation_lightgbm_fold{fold_number}.parquet')
print('Validation predictions saved.')

### 3. Final model and test set prediction

In [ ]:
test_df = read_parquet_file(data_dir + 'processed/test.parquet')

In [ ]:
# train a new model on the entire train set (Jan 2020 - Feb 2021) using the best hyperparams from Optuna
model = LightGBMModel(**study.best_params)
model.fit(train_validation_df)

# do predictions on the test set (Mar -April 2021)
test_df[PREDICTION] = model.predict(test_df)

# save the model and results
model.save_model(model_dir + 'lightgbm_model.pkl')
save_to_parquet_file(test_df, data_dir + 'predictions/test_lightgbm.parquet')
print('Model and test predictions saved.')

In [ ]:
# test set scores (for reporting only: the test set was not used to choose any hyperparameter)
test_scores = evaluate(model, test_df)
print({name: round(score, 4) for name, score in test_scores.items()})

In [ ]:
test_df.head()